# Scraping Berita Saham — KONTAN (REVISI v2)

Notebook ini menyusun ulang scraper **Kontan.co.id** dengan format dan konvensi yang
**identik** dengan scraper Detik, ANTARA, Bloomberg Technoz, dan CNBC Indonesia,
sehingga kelima CSV keluarannya dapat digabung langsung pada PREP_02.

### Perbaikan audit yang sudah tertanam
1. **Dedup `['Link', 'Kode_Emiten']`** — artikel yang relevan untuk dua emiten tidak terbuang.
2. **Pencocokan alias berbatas kata** (`_sebut`) — cegah *false positive* alias pendek.
3. **429 tidak masuk `status_forcelist`** — rate-limit ditangani sepenuhnya oleh `safe_get()`.
4. **Periode 1 Jan 2022 – 30 Jun 2026** — seragam dengan empat sumber lain.

> ⚠️ **WAJIB: jalankan PROBE (Bagian 6) lebih dulu.** Struktur HTML & skema paginasi
> Kontan diverifikasi otomatis di sana. Bila probe gagal, perbaiki selector sebelum
> menjalankan Bagian 8 — jangan langsung scraping penuh.

## 1. Install Library

In [2]:
!pip install -q tqdm requests beautifulsoup4 lxml pandas


[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import Library & Konfigurasi

In [3]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup as bs
from datetime import datetime, timedelta, timezone
import time, random, re
import pandas as pd
from pathlib import Path
from tqdm import tqdm

WIB = timezone(timedelta(hours=7))

start_date = datetime(2022, 1, 1, tzinfo=WIB)
end_date   = datetime(2026, 6, 30, 23, 59, 59, tzinfo=WIB)

OUTPUT_DIR           = Path('../../03_dataset/raw')
OUTPUT_DIR_DASHBOARD = Path('../../dashboard/data')
CHECKPOINT_DIR       = Path('checkpoint_kontan')
for d in (OUTPUT_DIR, OUTPUT_DIR_DASHBOARD, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / 'berita_kontan_lq45_energi.csv'
BASE_URL    = 'https://www.kontan.co.id'

print("Konfigurasi siap")
print(f"Periode : {start_date:%d %B %Y} s/d {end_date:%d %B %Y}")
print(f"Output  : {OUTPUT_FILE}")

Konfigurasi siap
Periode : 01 January 2022 s/d 30 June 2026
Output  : ..\..\03_dataset\raw\berita_kontan_lq45_energi.csv


## 3. Daftar 7 Emiten: Keyword Pencarian vs Alias Filter

Dua kamus dipisah dengan sengaja:
- **`SEARCH_KEYWORDS`** → dilempar ke mesin pencari Kontan. Pendek (1–2 kata) demi *recall* tinggi.
- **`EMITEN_ALIAS`** → dipakai filter relevansi untuk memastikan artikel benar-benar menyebut emiten.

In [4]:
# Keyword PENCARIAN -> recall tinggi, biar filter yang menyaring
SEARCH_KEYWORDS = {
    'ADRO': ['Adaro', 'Alamtri', 'ADRO'],
    'ADMR': ['Adaro Minerals', 'ADMR'],
    'ITMG': ['Indo Tambangraya', 'ITMG'],
    'PTBA': ['Bukit Asam', 'PTBA'],
    'MEDC': ['Medco', 'MEDC'],
    'AKRA': ['AKR Corporindo', 'AKRA'],
    'PGAS': ['PGN', 'Perusahaan Gas Negara', 'PGAS'],
}

# Alias untuk FILTER relevansi. Sejak revisi v2, pencocokan memakai BATAS KATA
# (_sebut) sehingga alias pendek TIDAK perlu lagi diberi spasi di belakang.
EMITEN_ALIAS = {
    'ADRO': ['adro', 'adaro energy', 'adaro', 'alamtri'],
    'ADMR': ['admr', 'adaro minerals'],
    'ITMG': ['itmg', 'indo tambangraya', 'tambangraya'],
    'PTBA': ['ptba', 'bukit asam'],
    'MEDC': ['medc', 'medco'],
    'AKRA': ['akra', 'akr corporindo'],
    'PGAS': ['pgas', 'perusahaan gas negara', 'pgn'],
}

NAMA_PERUSAHAAN = {
    'ADRO': 'Adaro Energy Indonesia Tbk (Alamtri Resources Indonesia Tbk)',
    'ADMR': 'Adaro Minerals Indonesia Tbk',
    'ITMG': 'Indo Tambangraya Megah Tbk',
    'PTBA': 'Bukit Asam Tbk',
    'MEDC': 'Medco Energi Internasional Tbk',
    'AKRA': 'AKR Corporindo Tbk',
    'PGAS': 'Perusahaan Gas Negara Tbk',
}

URUTAN_EMITEN = list(SEARCH_KEYWORDS.keys())
print(f"Total emiten           : {len(SEARCH_KEYWORDS)}")
print(f"Total keyword pencarian: {sum(len(v) for v in SEARCH_KEYWORDS.values())}")
for k, v in SEARCH_KEYWORDS.items():
    print(f"  {k}: {v}")

Total emiten           : 7
Total keyword pencarian: 16
  ADRO: ['Adaro', 'Alamtri', 'ADRO']
  ADMR: ['Adaro Minerals', 'ADMR']
  ITMG: ['Indo Tambangraya', 'ITMG']
  PTBA: ['Bukit Asam', 'PTBA']
  MEDC: ['Medco', 'MEDC']
  AKRA: ['AKR Corporindo', 'AKRA']
  PGAS: ['PGN', 'Perusahaan Gas Negara', 'PGAS']


## 4. Filter Relevansi (pencocokan berbatas kata)

In [5]:
# ══════════════════════════════════════════════════════════════════════════
# PENCOCOKAN ALIAS DENGAN BATAS KATA (WORD BOUNDARY) — standar audit v2
# ══════════════════════════════════════════════════════════════════════════
# Substring polos (`alias in teks`) membuat alias pendek ikut cocok di dalam
# kata lain ('pgn' pada "pgnXYZ", 'akr' pada "akronim") -> false positive.
# ══════════════════════════════════════════════════════════════════════════
import re as _re

def _sebut(alias_list, teks):
    """True bila salah satu alias muncul sebagai KATA UTUH di `teks`."""
    for a in alias_list:
        a = str(a).strip()
        if a and _re.search(rf'(?<!\w){_re.escape(a)}(?!\w)', teks):
            return True
    return False

def _hitung_sebut(alias_list, teks):
    """Jumlah kemunculan alias sebagai kata utuh (maksimum antar alias)."""
    return max((len(_re.findall(rf'(?<!\w){_re.escape(str(a).strip())}(?!\w)', teks))
                for a in alias_list if str(a).strip()), default=0)

# ── F1: URL non-artikel yang dibuang ──
TIPE_BUANG_URL = [
    '/foto/', '/video/', '/galeri/', '/infografik/', '/tag/', '/topik/',
    '/momsmoney', '/lifestyle/', '/hiburan/', '/olahraga/', '/kesehatan/',
    'peluangusaha.kontan', 'lifestyle.kontan', 'sportsmania.kontan',
]
def filter_url(link):
    l = str(link).lower()
    if 'kontan.co.id' not in l:
        return False
    if '/news/' not in l:          # seluruh artikel Kontan memakai pola /news/
        return False
    return not any(t in l for t in TIPE_BUANG_URL)

# ── F2: blacklist headline (politik TIDAK diblacklist — bisa relevan ke volatilitas) ──
KATA_BUANG_HEADLINE = [
    'wisata', 'liburan', 'hotel', 'kuliner', 'resep', 'makanan',
    'fashion', 'skincare', 'film', 'artis', 'gosip', 'musik',
    'konser', 'drakor', 'bola', 'liga', 'pertandingan',
    'tips kesehatan', 'penyakit', 'vaksin', 'horoskop', 'zodiak',
    'promo', 'diskon', 'giveaway', 'kode redeem',
]
def filter_headline(headline):
    h = str(headline).lower()
    return not any(k in h for k in KATA_BUANG_HEADLINE)

# ── Kata konteks pasar ──
KATA_KONTEKS_SAHAM = [
    'saham', 'bursa', 'bei', 'idx', 'ihsg', 'emiten', 'investor',
    'laba', 'rugi', 'pendapatan', 'revenue', 'kinerja', 'keuangan',
    'kuartal', 'dividen', 'buyback', 'merger', 'akuisisi',
    'rights issue', 'right issue', 'ipo', 'target harga', 'rekomendasi',
    'analis', 'sekuritas', 'valuasi', 'harga saham', 'koreksi',
    'menguat', 'melemah', 'melesat', 'produksi', 'batubara', 'batu bara',
    'minyak', 'gas', 'lifting', 'laba bersih', 'net profit',
    'pasar modal', 'obligasi', 'rupst', 'capex', 'ekspor', 'penjualan',
]

def lolos_relevansi(kode, headline, isi):
    """Return (bool, alasan). Gerbang wajib: emiten disebut sebagai kata utuh."""
    teks_full = (str(headline) + ' ' + str(isi)).lower()
    alias = EMITEN_ALIAS.get(kode, [kode.lower()])

    if not _sebut(alias, teks_full):
        return False, 'tidak_sebut_emiten'

    n_konteks = sum(1 for k in KATA_KONTEKS_SAHAM if k in teks_full)
    sebut_di_headline = _sebut(alias, str(headline).lower())

    if n_konteks >= 1 or sebut_di_headline:
        return True, f'ok(konteks={n_konteks})'
    return False, 'emiten_disebut_konteks_lemah'

print("Filter F1, F2, dan lolos_relevansi() siap")
print("  Uji cepat batas kata:")
print("   'pgn' pada 'pgnXYZ'          ->", _sebut(['pgn'], 'pgnxyz'), "(harus False)")
print("   'pgn' pada 'laba PGN naik'   ->", _sebut(['pgn'], 'laba pgn naik'), "(harus True)")

Filter F1, F2, dan lolos_relevansi() siap
  Uji cepat batas kata:
   'pgn' pada 'pgnXYZ'          -> False (harus False)
   'pgn' pada 'laba PGN naik'   -> True (harus True)


## 5. HTTP Session, Parser Tanggal, Ekstraktor Body & Tanggal

In [6]:
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0',
]
def get_headers():
    return {
        'User-Agent'     : random.choice(USER_AGENTS),
        'Accept'         : 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer'        : 'https://www.kontan.co.id/',
    }

def buat_session():
    s = requests.Session()
    # FIX AUDIT: 429 SENGAJA TIDAK dimasukkan ke status_forcelist. Bila 429
    # ditangani urllib3 (backoff_factor=2) DAN loop manual safe_get() sekaligus,
    # jedanya berlipat -> run bisa macet berjam-jam. Rate-limit ditangani
    # SEPENUHNYA oleh safe_get() di bawah.
    retry = Retry(total=4, backoff_factor=2, status_forcelist=[500, 502, 503, 504])
    s.mount('https://', HTTPAdapter(max_retries=retry))
    return s
sess = buat_session()

def safe_get(url, timeout=25, max_retry=3):
    global sess
    for attempt in range(1, max_retry + 1):
        try:
            r = sess.get(url, headers=get_headers(), timeout=timeout)
            if r.status_code == 429:
                wait = 60 * attempt
                print(f"  Rate-limit 429 - tunggu {wait}s..."); time.sleep(wait)
                sess = buat_session(); continue
            return r
        except requests.exceptions.ConnectionError as e:
            print(f"  ConnectionError ({attempt}/{max_retry}): {str(e)[:60]}")
            time.sleep(10 * attempt); sess = buat_session()
        except requests.exceptions.Timeout:
            time.sleep(5 * attempt)
        except Exception as e:
            print(f"  Error: {e}"); return None
    return None

# ── Parser tanggal: Kontan memakai format Indonesia "Senin, 15 Juli 2026 / 10:30 WIB" ──
BULAN_ID = {
    'januari':1, 'februari':2, 'maret':3, 'april':4, 'mei':5, 'juni':6,
    'juli':7, 'agustus':8, 'september':9, 'oktober':10, 'november':11, 'desember':12,
    'jan':1, 'feb':2, 'mar':3, 'apr':4, 'jun':6, 'jul':7, 'agu':8, 'ags':8,
    'sep':9, 'okt':10, 'nov':11, 'des':12,
}

def parse_date_kontan(raw):
    """Terima ISO (meta tag) maupun teks Indonesia. Return datetime tz-aware / None."""
    if not raw:
        return None
    raw = str(raw).strip()

    # 1) ISO 8601 dari <meta property="article:published_time">
    try:
        dt = datetime.fromisoformat(raw.replace('Z', '+00:00'))
        return dt if dt.tzinfo else dt.replace(tzinfo=WIB)
    except Exception:
        pass
    m = re.search(r'(\d{4}-\d{2}-\d{2})[T ](\d{2}:\d{2})', raw)
    if m:
        try:
            return datetime.fromisoformat(f'{m.group(1)} {m.group(2)}').replace(tzinfo=WIB)
        except Exception:
            pass

    # 2) Teks Indonesia: "15 Juli 2026 / 10:30 WIB" (hari & "WIB" opsional)
    low = raw.lower()
    m = re.search(r'(\d{1,2})\s+([a-z]+)\s+(\d{4})', low)
    if m:
        hari, nama_bulan, tahun = int(m.group(1)), m.group(2), int(m.group(3))
        bulan = BULAN_ID.get(nama_bulan)
        if bulan:
            jam, menit = 0, 0
            mt = re.search(r'(\d{1,2})[:.](\d{2})', low[m.end():])
            if mt:
                jam, menit = int(mt.group(1)), int(mt.group(2))
            try:
                return datetime(tahun, bulan, hari, jam, menit, tzinfo=WIB)
            except ValueError:
                return None
    return None

NOISE_KONTAN = [
    'Baca Juga:', 'Baca juga:', 'BACA JUGA:', 'Reporter:', 'Editor:',
    'KONTAN.CO.ID', 'Kontan.co.id', 'Selanjutnya:', 'Cek Berita dan Artikel',
    'Video Pilihan', 'TAG:', 'Bagikan',
]
def clean_text(text):
    for n in NOISE_KONTAN:
        text = text.replace(n, '')
    return re.sub(r'\s{2,}', ' ', text).strip()

# Selector body kandidat. DETECTED_BODY_SEL diisi oleh Probe (Bagian 6).
BODY_SELECTORS = [
    'div.tmpt-desk-kon', 'div[itemprop="articleBody"]', 'div.detail-desk',
    'div.detail-berita', 'div.isi-berita', 'div.kontan-detail',
    'div.article-content', 'article',
]
DETECTED_BODY_SEL = None

def _body_from_soup(soup):
    sels = ([DETECTED_BODY_SEL] if DETECTED_BODY_SEL else []) + BODY_SELECTORS
    for sel in sels:
        if not sel:
            continue
        el = soup.select_one(sel)
        if el:
            ps = el.find_all('p')
            txt = clean_text(' '.join(p.get_text(strip=True) for p in ps)) if ps \
                  else clean_text(el.get_text(' ', strip=True))
            if len(txt) > 300:
                return txt
    # Fallback: gabung <p> panjang (>40 char) di seluruh halaman
    allp = [p.get_text(strip=True) for p in soup.find_all('p')]
    return clean_text(' '.join(p for p in allp if len(p) > 40))

def _date_from_soup(soup):
    for prop in ['article:published_time', 'og:published_time', 'article:modified_time']:
        meta = soup.find('meta', {'property': prop}) or soup.find('meta', {'name': prop})
        if meta and meta.get('content'):
            dt = parse_date_kontan(meta['content'])
            if dt:
                return dt
    for sel in ['div.fs14.ff-open.font-gray', 'div.tanggal', 'span.font-gray',
                'div.detail-warna', 'time']:
        el = soup.select_one(sel)
        if el:
            dt = parse_date_kontan(el.get_text(' ', strip=True))
            if dt:
                return dt
    # Fallback terakhir: cari pola tanggal Indonesia di seluruh teks header
    dt = parse_date_kontan(soup.get_text(' ', strip=True)[:600])
    return dt

def fetch_article(link):
    """Return (datetime|None, body_str)."""
    r = safe_get(link)
    if not r or r.status_code != 200:
        return None, ''
    soup = bs(r.text, 'lxml')
    return _date_from_soup(soup), _body_from_soup(soup)

print("Session, parser tanggal, dan ekstraktor body siap")
print("  Uji parser tanggal:")
for s in ['2026-07-15T10:30:00+07:00', 'Senin, 15 Juli 2026 / 10:30 WIB',
          '03 Januari 2022 | 08:05 WIB', 'teks tanpa tanggal']:
    print(f"   {s[:40]:42} -> {parse_date_kontan(s)}")

Session, parser tanggal, dan ekstraktor body siap
  Uji parser tanggal:
   2026-07-15T10:30:00+07:00                  -> 2026-07-15 10:30:00+07:00
   Senin, 15 Juli 2026 / 10:30 WIB            -> 2026-07-15 10:30:00+07:00
   03 Januari 2022 | 08:05 WIB                -> 2022-01-03 08:05:00+07:00
   teks tanpa tanggal                         -> None


## 6. PROBE: Auto-deteksi Skema Pagination & Selector Body

**Jalankan sel ini lebih dulu.** Sel ini menguji tiga kemungkinan skema paginasi
pencarian Kontan dan memilih yang benar-benar menghasilkan artikel baru di halaman
kedua, lalu mendeteksi selector body dari satu artikel contoh.

In [7]:
import requests as _rq

def build_search_url(kw, scheme, idx):
    """idx = 0,1,2,... (halaman ke-(idx+1))."""
    q = _rq.utils.quote(kw)
    if scheme == 'per_page':      # skema historis Kontan: offset kelipatan 20
        return f'{BASE_URL}/search/?search={q}&per_page={idx * 20}'
    if scheme == 'page_query':
        return f'{BASE_URL}/search/?search={q}&page={idx + 1}'
    if scheme == 'pos_indeks':
        return f'{BASE_URL}/search/?search={q}&pos=indeks&per_page={idx * 20}'
    raise ValueError(scheme)

def extract_listing(soup):
    """Ambil [(href, title)] unik dari halaman hasil pencarian."""
    out, seen_local = [], set()
    for a in soup.find_all('a', href=True):
        href, title = a['href'], a.get_text(strip=True)
        if '/news/' not in href:
            continue
        if not href.startswith('http'):
            href = ('https:' + href) if href.startswith('//') else (BASE_URL + href)
        if 'kontan.co.id' not in href or len(title) < 10:
            continue
        if href not in seen_local:
            seen_local.add(href); out.append((href, title))
    return out

def detect_scheme(kw='Adaro'):
    print(f"  Menguji skema pagination dengan keyword '{kw}'...")
    for scheme in ['per_page', 'page_query', 'pos_indeks']:
        try:
            r1 = safe_get(build_search_url(kw, scheme, 0))
            r2 = safe_get(build_search_url(kw, scheme, 1))
            if not r1 or not r2:
                print(f"    [{scheme:10}] gagal ambil halaman"); continue
            s1 = {h for h, _ in extract_listing(bs(r1.text, 'lxml'))}
            s2 = {h for h, _ in extract_listing(bs(r2.text, 'lxml'))}
            baru = s2 - s1
            status = 'VALID' if (s1 and baru) else 'tidak jalan'
            print(f"    [{scheme:10}] hal1={len(s1):2d} | hal2={len(s2):2d} | baru={len(baru):2d} -> {status}")
            if s1 and baru:
                return scheme
        except Exception as e:
            print(f"    [{scheme:10}] error: {str(e)[:50]}")
    print("    -> Tidak ada skema yang menambah hasil. Fallback 'per_page' (1 halaman saja).")
    return 'per_page'

def detect_body_selector(sample_url):
    global DETECTED_BODY_SEL
    r = safe_get(sample_url)
    if not r:
        print("  Gagal mengambil artikel contoh."); return None
    soup = bs(r.text, 'lxml')
    print(f"  Menguji selector body pada: {sample_url[:70]}")
    best, best_n = None, 0
    for sel in BODY_SELECTORS:
        el = soup.select_one(sel)
        n = len(el.get_text(strip=True)) if el else 0
        if n > best_n:
            best, best_n = sel, n
        mark = '  <- kandidat' if n > 400 else ''
        print(f"    {sel:34} : {n:5d} char{mark}")
    if best_n > 400:
        DETECTED_BODY_SEL = best
        print(f"  -> Selector body terdeteksi: '{best}' ({best_n} char)")
    else:
        print("  -> Tidak ada selector >400 char; scraper akan pakai fallback klaster <p>.")
    print(f"  -> Uji tanggal artikel contoh: {_date_from_soup(soup)}")
    return DETECTED_BODY_SEL

# Jalankan deteksi
PAGINATION_SCHEME = detect_scheme('Adaro')
print(f"\n>> SKEMA PAGINATION DIPAKAI: {PAGINATION_SCHEME}\n")

_r = safe_get(build_search_url('Adaro', PAGINATION_SCHEME, 0))
_contoh = extract_listing(bs(_r.text, 'lxml')) if _r else []
if _contoh:
    detect_body_selector(_contoh[0][0])
else:
    print("Tidak ada artikel contoh untuk uji selector. Cek koneksi / struktur situs.")

  Menguji skema pagination dengan keyword 'Adaro'...
    [per_page  ] hal1=38 | hal2=38 | baru=20 -> VALID

>> SKEMA PAGINATION DIPAKAI: per_page

  Menguji selector body pada: https://investasi.kontan.co.id/news/adaro-andalan-aadi-kucurkan-pinjam
    div.tmpt-desk-kon                  :  3009 char  <- kandidat
    div[itemprop="articleBody"]        :  3009 char  <- kandidat
    div.detail-desk                    :     0 char
    div.detail-berita                  :     0 char
    div.isi-berita                     :     0 char
    div.kontan-detail                  :     0 char
    div.article-content                :     0 char
    article                            :    87 char
  -> Selector body terdeteksi: 'div.tmpt-desk-kon' (3009 char)
  -> Uji tanggal artikel contoh: 2026-07-05 09:26:58+07:00


## 7. Scraper Utama (pagination robust + berhenti adaptif)

In [8]:
def scrape_kontan(kode, keywords, max_pages=40, patience=2,
                  d_min=1.5, d_max=3.0, checkpoint_dir=CHECKPOINT_DIR):
    checkpoint_file = checkpoint_dir / f'{kode}_checkpoint.csv'
    if checkpoint_file.exists():
        df_ckpt = pd.read_csv(checkpoint_file, encoding='utf-8-sig')
        results = df_ckpt.to_dict('records')
        seen    = set(df_ckpt['Link'].tolist())
        print(f'  Resume checkpoint: {len(results)} artikel')
    else:
        results, seen = [], set()

    stat = dict(kandidat=0, lolos_f1=0, lolos_f2=0, disimpan=0,
                out_of_range=0, body_kosong=0, tanggal_gagal=0,
                ditolak_relevansi=0, duplikat=0)

    for kw in keywords:
        print(f'  [{kode}|KONTAN] keyword: "{kw}"')
        empty_streak = 0

        for page_idx in range(max_pages):
            url = build_search_url(kw, PAGINATION_SCHEME, page_idx)
            r = safe_get(url)
            if not r:
                empty_streak += 1
                if empty_streak >= patience: break
                time.sleep(random.uniform(d_min, d_max)); continue

            listing = extract_listing(bs(r.text, 'lxml'))
            baru = [(h, t) for h, t in listing if h not in seen]

            if not baru:
                empty_streak += 1
                if empty_streak >= patience:
                    print(f'    Halaman {page_idx+1}: {patience}x tanpa link baru -> stop keyword')
                    break
                time.sleep(random.uniform(d_min, d_max)); continue
            empty_streak = 0

            for link, title in tqdm(baru, desc=f'    [{kode}] hal {page_idx+1}',
                                    unit='art', leave=False):
                if link in seen:
                    stat['duplikat'] += 1; continue
                stat['kandidat'] += 1
                seen.add(link)

                if not filter_url(link):       continue
                stat['lolos_f1'] += 1
                if not filter_headline(title): continue
                stat['lolos_f2'] += 1

                time.sleep(random.uniform(d_min, d_max))
                dt, body = fetch_article(link)
                if not dt:
                    stat['tanggal_gagal'] += 1; continue
                if not (start_date <= dt <= end_date):
                    stat['out_of_range'] += 1; continue
                if not body.strip():
                    stat['body_kosong'] += 1; continue

                ok, alasan = lolos_relevansi(kode, title, body)
                if not ok:
                    stat['ditolak_relevansi'] += 1; continue

                results.append({
                    'Sumber': 'Kontan', 'Kode_Emiten': kode, 'Keyword': kw,
                    'Headline': title, 'Tanggal': dt.strftime('%Y-%m-%d %H:%M'),
                    'Link': link, 'Isi_Berita': body,
                })
                stat['disimpan'] += 1
                tqdm.write(f'    [{stat["disimpan"]}] {title[:70]}')
                if stat['disimpan'] % 25 == 0:
                    pd.DataFrame(results).to_csv(checkpoint_file, index=False, encoding='utf-8-sig')

            time.sleep(random.uniform(d_min, d_max))

    if results:
        pd.DataFrame(results).to_csv(checkpoint_file, index=False, encoding='utf-8-sig')

    print(f'\n  STATISTIK [{kode}]  ' + '-'*30)
    for k, v in stat.items():
        print(f'    {k:20}: {v:>6,}')
    return results

print("scrape_kontan() siap")

scrape_kontan() siap


## 8. Jalankan Scraping Semua Emiten

In [9]:
assert 'PAGINATION_SCHEME' in dir(), "Jalankan PROBE (Bagian 6) lebih dulu!"

semua_artikel = []
mulai_jam = datetime.now(WIB)

for kode, keywords in SEARCH_KEYWORDS.items():
    print("\n" + "=" * 65)
    print(f"  {kode} | KONTAN | {start_date:%b %Y} - {end_date:%b %Y}")
    print(f"  Mulai: {datetime.now(WIB):%H:%M:%S} | Keyword: {len(keywords)}")
    print("=" * 65)

    hasil = scrape_kontan(kode, keywords)
    semua_artikel.extend(hasil)

    # FIX AUDIT: dedup pada PASANGAN (Link, Kode_Emiten) — satu artikel yang
    # relevan untuk dua emiten sah disimpan dua kali dengan Kode_Emiten berbeda.
    df_tmp = pd.DataFrame(semua_artikel).drop_duplicates(subset=['Link', 'Kode_Emiten'])
    df_tmp.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    elapsed = (datetime.now(WIB) - mulai_jam).total_seconds() / 60
    print(f"  Auto-save: {len(df_tmp):,} artikel | {elapsed:.1f} menit")
    time.sleep(random.uniform(8, 15))

df_final = (pd.DataFrame(semua_artikel)
              .drop_duplicates(subset=['Link', 'Kode_Emiten'])
              .reset_index(drop=True))
df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

total = (datetime.now(WIB) - mulai_jam).total_seconds() / 60
print("\n" + "=" * 65)
print(f"  SELESAI | {total:.1f} menit | {len(df_final):,} artikel")
print("=" * 65)
if len(df_final):
    print(df_final['Kode_Emiten'].value_counts().reindex(URUTAN_EMITEN).to_string())


  ADRO | KONTAN | Jan 2022 - Jun 2026
  Mulai: 07:13:07 | Keyword: 3
  Resume checkpoint: 118 artikel
  [ADRO|KONTAN] keyword: "Adaro"


    [ADRO] hal 1:   5%|▌         | 2/38 [00:05<01:40,  2.80s/art]

    [1] Harga AADI Turun 30% dari Rekor Tertinggi, Fundamental Melemah atau Pa


    [ADRO] hal 1:   8%|▊         | 3/38 [00:07<01:27,  2.50s/art]

    [2] Meski Kinerja Tersendat pada Kuartal I, Prospek AADI pada Tahun 2026 M


    [ADRO] hal 1:  11%|█         | 4/38 [00:10<01:32,  2.72s/art]

... (10,124 outputs left out)

    [1098] Bursa Jumat (14/1) Segera Dibuka, Saham Ini Diprediksi Akan Bullish


    [PGAS] hal 38:  93%|█████████▎| 13/14 [00:31<00:02,  2.49s/art]

    [1099] Simak Rekomendasi Teknikal Saham BBNI, INKP, dan PGAS untuk Perdaganga



  STATISTIK [PGAS]  ------------------------------
    kandidat            :  1,699
    lolos_f1            :  1,693
    lolos_f2            :  1,675
    disimpan            :  1,099
    out_of_range        :    554
    body_kosong         :      0
    tanggal_gagal       :      3
    ditolak_relevansi   :     19
    duplikat            :      0
  Auto-save: 5,911 artikel | 338.8 menit

  SELESAI | 339.0 menit | 5,911 artikel
Kode_Emiten
ADRO    1167
ADMR     425
ITMG     563
PTBA     926
MEDC     891
AKRA     733
PGAS    1206


## 9. Rekap & Validasi Hasil Akhir

In [10]:
if 'df_final' not in dir() or df_final is None or len(df_final) == 0:
    df_final = pd.read_csv(OUTPUT_FILE, encoding='utf-8-sig')

df_final['Tanggal'] = pd.to_datetime(df_final['Tanggal'])

print("=" * 65)
print("  REKAP HASIL SCRAPING — KONTAN")
print("=" * 65)
print(f"Total artikel  : {len(df_final):,}")
print(f"Rentang tanggal: {df_final['Tanggal'].min():%Y-%m-%d} s/d {df_final['Tanggal'].max():%Y-%m-%d}")

print("\nPer emiten:")
print(df_final['Kode_Emiten'].value_counts().reindex(URUTAN_EMITEN).to_string())

print("\nPer tahun x emiten:")
print(pd.crosstab(df_final['Tanggal'].dt.year, df_final['Kode_Emiten'])
        .reindex(columns=URUTAN_EMITEN, fill_value=0).to_string())

# Indikator presisi: makin tinggi makin sedikit false positive
def _di_judul(r):
    return _sebut(EMITEN_ALIAS.get(r['Kode_Emiten'], []), str(r['Headline']).lower())
prop = df_final.apply(_di_judul, axis=1).mean() * 100
print(f"\nProporsi emiten disebut di judul: {prop:.0f}% (makin tinggi makin presisi)")

# Validasi mutu data
print("\nValidasi:")
print(f"  Duplikat (Link,Emiten) : {df_final.duplicated(subset=['Link','Kode_Emiten']).sum()}")
print(f"  Isi_Berita kosong      : {(df_final['Isi_Berita'].fillna('').str.len() < 100).sum()}")
print(f"  Di luar periode        : {((df_final['Tanggal'] < start_date.replace(tzinfo=None)) | (df_final['Tanggal'] > end_date.replace(tzinfo=None))).sum()}")

  REKAP HASIL SCRAPING — KONTAN
Total artikel  : 5,911
Rentang tanggal: 2022-01-03 s/d 2026-06-30

Per emiten:
Kode_Emiten
ADRO    1167
ADMR     425
ITMG     563
PTBA     926
MEDC     891
AKRA     733
PGAS    1206

Per tahun x emiten:
Kode_Emiten  ADRO  ADMR  ITMG  PTBA  MEDC  AKRA  PGAS
Tanggal                                              
2022          212    95   172   184    78   168   227
2023          281    94   169   279   288   228   258
2024          361    72    98   212   218   154   285
2025          207   111    88   176   216   146   316
2026          106    53    36    75    91    37   120

Proporsi emiten disebut di judul: 88% (makin tinggi makin presisi)

Validasi:
  Duplikat (Link,Emiten) : 0
  Isi_Berita kosong      : 0
  Di luar periode        : 0


## 10. Simpan Output dengan Kolom Standar

In [11]:
KOLOM_STANDAR = ['Sumber', 'Kode_Emiten', 'Keyword', 'Headline',
                 'Tanggal', 'Link', 'Isi_Berita']

df_simpan = df_final.copy()
for kol in KOLOM_STANDAR:
    if kol not in df_simpan.columns:
        df_simpan[kol] = ''
df_simpan['Sumber']  = 'Kontan'
df_simpan['Tanggal'] = pd.to_datetime(df_simpan['Tanggal']).dt.strftime('%Y-%m-%d %H:%M')
df_simpan = (df_simpan[KOLOM_STANDAR]
               .drop_duplicates(subset=['Link', 'Kode_Emiten'])
               .sort_values('Tanggal')
               .reset_index(drop=True))

df_simpan.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

print("=" * 65)
print(f"  File tersimpan : {OUTPUT_FILE}")
print(f"  Total artikel  : {len(df_simpan):,}")
print(f"  Kolom          : {list(df_simpan.columns)}")
print("=" * 65)
print("LANGKAH SELANJUTNYA:")
print("  -> Gabungkan kelima CSV (Detik, ANTARA, Bloomberg Technoz, CNBC, Kontan)")
print("     pada notebook PREP_02 (Analisis Sentimen IndoBERT)")
print("=" * 65)
df_simpan.head(3)

  File tersimpan : ..\..\03_dataset\raw\berita_kontan_lq45_energi.csv
  Total artikel  : 5,911
  Kolom          : ['Sumber', 'Kode_Emiten', 'Keyword', 'Headline', 'Tanggal', 'Link', 'Isi_Berita']
LANGKAH SELANJUTNYA:
  -> Gabungkan kelima CSV (Detik, ANTARA, Bloomberg Technoz, CNBC, Kontan)
     pada notebook PREP_02 (Analisis Sentimen IndoBERT)


,Sumber,Kode_Emiten,Keyword,Headline,Tanggal,Link,Isi_Berita
0,Kontan,AKRA,AKRA,"Prediksi IHSG Menguat Di Awal Tahun 2022, Ini ...",2022-01-03 08:08,https://investasi.kontan.co.id/news/prediksi-i...,Kenia Intan| Adi Wikanto - JAKARTA.Bursa Efek ...
1,Kontan,ADMR,Adaro Minerals,"Resmi Melantai, Saham Adaro Mineral (ADMR) Mel...",2022-01-03 10:08,https://investasi.kontan.co.id/news/resmi-mela...,Akhmad Suryahadi| Wahyu T.Rahmawati - JAKARTA....
2,Kontan,ADRO,Adaro,"Resmi Melantai, Saham Adaro Mineral (ADMR) Mel...",2022-01-03 10:08,https://investasi.kontan.co.id/news/resmi-mela...,Akhmad Suryahadi| Wahyu T.Rahmawati - JAKARTA....
